In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install wandb

import torch
import torch.nn as nn
import numpy as np
import math
import wandb
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

paths = {
    "NURBS": (
        "/content/drive/MyDrive/LDC_data/nurbs_lid_driven_cavity_X.npz",
        "/content/drive/MyDrive/LDC_data/nurbs_lid_driven_cavity_Y.npz"
    ),
    "Harmonics": (
        "/content/drive/MyDrive/LDC_data/harmonics_lid_driven_cavity_X.npz",
        "/content/drive/MyDrive/LDC_data/harmonics_lid_driven_cavity_Y.npz"
    ),
    "Skelneton": (
        "/content/drive/MyDrive/LDC_data/skelneton_lid_driven_cavity_X.npz",
        "/content/drive/MyDrive/LDC_data/skelneton_lid_driven_cavity_Y.npz"
    )
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


In [ ]:
def create_coords(H, W, device):
    """
    Creates normalized (x, y) coordinates in [0, 1]^2.
    Output shape: [H*W, 2]
    SIREN handles spatial frequency internally via omega_0.
    """
    x = torch.linspace(0, 1, H)
    y = torch.linspace(0, 1, W)
    grid_x, grid_y = torch.meshgrid(x, y, indexing='ij')
    coords = torch.stack([grid_x, grid_y], dim=-1)
    return coords.reshape(-1, 2).to(device)   # [H*W, 2]

In [ ]:
class SineLayer(nn.Module):
    def __init__(self, in_features, out_features, bias=True,
                 is_first=False, omega_0=30.0):
        super().__init__()
        self.omega_0     = omega_0
        self.is_first    = is_first
        self.in_features = in_features
        self.linear      = nn.Linear(in_features, out_features, bias=bias)
        self.init_weights()

    def init_weights(self):
        with torch.no_grad():
            if self.is_first:
                self.linear.weight.uniform_(
                    -1 / self.in_features,
                     1 / self.in_features
                )
            else:
                bound = np.sqrt(6 / self.in_features) / self.omega_0
                self.linear.weight.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class SirenNet(nn.Module):
    def __init__(self, in_dim, hidden=512, out_dim=512,
                 num_layers=6, first_omega_0=30.0, hidden_omega_0=30.0):
        super().__init__()
        self.net = nn.Sequential()

        # First layer
        self.net.add_module(
            "layer_0",
            SineLayer(in_dim, hidden, is_first=True, omega_0=first_omega_0)
        )

        # Hidden layers
        for i in range(1, num_layers - 1):
            self.net.add_module(
                f"layer_{i}",
                SineLayer(hidden, hidden, is_first=False, omega_0=hidden_omega_0)
            )

        # Final linear layer
        final_linear = nn.Linear(hidden, out_dim, bias=True)
        with torch.no_grad():
            bound = np.sqrt(6 / hidden) / hidden_omega_0
            final_linear.weight.uniform_(-bound, bound)
            nn.init.zeros_(final_linear.bias)
        self.net.add_module(f"layer_{num_layers - 1}", final_linear)

    def forward(self, x):
        return self.net(x)

In [ ]:
class DeepONet(nn.Module):
    """
    DeepONet with pure SIREN branch and trunk.
    Trunk input: raw (x, y) coords in [0,1]^2  — shape [N, 2]
    Branch input: flattened + normalized input field — shape [B, C*H*W]
    """
    def __init__(self, branch_dim, latent_dim=512):
        super().__init__()
        self.latent_dim = latent_dim

        # Branch: omega_0=1.0 (input is field values, not coordinates)
        self.branch = SirenNet(
            in_dim=branch_dim,
            hidden=512,
            out_dim=latent_dim,
            num_layers=6,
            first_omega_0=1.0,
            hidden_omega_0=1.0
        )

        # Trunk: omega_0=30.0 (input is spatial coordinates)
        # out_dim = latent_dim * 3 for three output channels (u, v, p)
        self.trunk = SirenNet(
            in_dim=2,
            hidden=512,
            out_dim=latent_dim * 3,
            num_layers=6,
            first_omega_0=30.0,
            hidden_omega_0=30.0
        )

        # Learnable per-channel output bias
        self.bias = nn.Parameter(torch.zeros(3))

    def forward(self, branch_input, coords):
        branch_out = self.branch(branch_input)             # [B, latent]
        trunk_out  = self.trunk(coords)                    # [N, 3*latent]
        trunk_out  = trunk_out.view(-1, 3, self.latent_dim)  # [N, 3, latent]

        # output[b, n, c] = sum_k branch[b,k] * trunk[n,c,k]
        output = torch.einsum("bk,nck->bnc", branch_out, trunk_out)
        output = output + self.bias                        # [B, N, 3]
        return output

In [ ]:
class Normalizer:
    """Z-score normalizer fitted on training data only."""
    def __init__(self, data, eps=1e-8):
        self.mean = data.mean(dim=0, keepdim=True)
        self.std  = data.std(dim=0, keepdim=True).clamp(min=eps)

    def encode(self, x):
        return (x - self.mean.to(x.device)) / self.std.to(x.device)

    def decode(self, x):
        return x * self.std.to(x.device) + self.mean.to(x.device)


def relative_l2(pred, true):
    """Global relative L2 — accumulate num/denom separately across batches."""
    return torch.norm(pred - true) / (torch.norm(true) + 1e-8)

In [ ]:
def train_geometry(geometry_name, x_path, y_path):
    print(f"\n==== Training on {geometry_name} ====")

    # ── Load data ─────────────────────────────────────────────────────────────
    X_data = np.load(x_path)["data"]
    Y_data = np.load(y_path)["data"]
    assert Y_data.shape[1] >= 3, "Y must have at least 3 channels (u, v, p)"
    Y_data = Y_data[:, 0:3, :, :]

    X = torch.tensor(X_data, dtype=torch.float32)
    Y = torch.tensor(Y_data, dtype=torch.float32)

    # ── Train / Val split ─────────────────────────────────────────────────────
    X_train, X_val, Y_train, Y_val = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )

    N, C, H, W = X_train.shape

    # ── Normalize inputs (fit on train only) ──────────────────────────────────
    X_flat_train = X_train.reshape(N, -1)
    X_flat_val   = X_val.reshape(X_val.shape[0], -1)

    x_norm       = Normalizer(X_flat_train)
    X_flat_train = x_norm.encode(X_flat_train)
    X_flat_val   = x_norm.encode(X_flat_val)

    # ── Normalize outputs ─────────────────────────────────────────────────────
    Y_flat_train = Y_train.reshape(Y_train.shape[0], 3, -1)  # [N, 3, H*W]
    y_norm       = Normalizer(Y_flat_train)

    # ── DataLoaders ───────────────────────────────────────────────────────────
    train_loader = DataLoader(
        TensorDataset(X_flat_train, Y_train),
        batch_size=16, shuffle=True, pin_memory=True
    )
    val_loader = DataLoader(
        TensorDataset(X_flat_val, Y_val),
        batch_size=16, shuffle=False, pin_memory=True
    )

    # ── Coords: pure (x,y) in [0,1]^2 ────────────────────────────────────────
    coords     = create_coords(H, W, device)   # [H*W, 2]
    branch_dim = C * H * W

    # ── Model ─────────────────────────────────────────────────────────────────
    model     = DeepONet(branch_dim=branch_dim, latent_dim=512).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    epochs    = 100
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )
    mse = nn.MSELoss()

    # ── W&B ───────────────────────────────────────────────────────────────────
    wandb.init(
        project="DeepONet-LDC",
        group="DeepONet-SIREN",
        name=f"{geometry_name}-run",
        config={
            "geometry":       geometry_name,
            "epochs":         epochs,
            "lr":             3e-4,
            "latent_dim":     512,
            "num_layers":     6,
            "hidden_dim":     512,
            "trunk_input":    "xy_coords",
            "resolution":     H,
            "activation":     "SIREN",
            "normalization":  "z-score"
        }
    )

    best_val_l2 = float("inf")
    save_path   = f"/content/drive/MyDrive/LDC_data/{geometry_name}_best.pt"

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        train_num, train_den = 0.0, 0.0

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            # Normalize targets
            yb_norm         = y_norm.encode(yb.reshape(yb.shape[0], 3, -1))
            yb_norm_spatial = yb_norm.permute(0, 2, 1)   # [B, H*W, 3]

            pred = model(xb, coords)                      # [B, H*W, 3]
            loss = mse(pred, yb_norm_spatial)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            train_num  += torch.norm(pred - yb_norm_spatial).item()
            train_den  += torch.norm(yb_norm_spatial).item()

        scheduler.step()
        train_loss = total_loss / len(train_loader)
        train_l2   = train_num / (train_den + 1e-8)

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        val_loss_sum = 0.0
        val_num, val_den = 0.0, 0.0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                yb_norm         = y_norm.encode(yb.reshape(yb.shape[0], 3, -1))
                yb_norm_spatial = yb_norm.permute(0, 2, 1)

                pred          = model(xb, coords)
                val_loss_sum += mse(pred, yb_norm_spatial).item()
                val_num      += torch.norm(pred - yb_norm_spatial).item()
                val_den      += torch.norm(yb_norm_spatial).item()

        val_loss = val_loss_sum / len(val_loader)
        val_l2   = val_num / (val_den + 1e-8)

        # ── Save best ─────────────────────────────────────────────────────────
        if val_l2 < best_val_l2:
            best_val_l2 = val_l2
            torch.save({
                "epoch":        epoch,
                "model":        model.state_dict(),
                "optimizer":    optimizer.state_dict(),
                "val_l2":       best_val_l2,
                "x_norm_mean":  x_norm.mean,
                "x_norm_std":   x_norm.std,
                "y_norm_mean":  y_norm.mean,
                "y_norm_std":   y_norm.std,
            }, save_path)

        wandb.log({
            "epoch":        epoch,
            "train_loss":   train_loss,
            "val_loss":     val_loss,
            "train_rel_L2": train_l2,
            "val_rel_L2":   val_l2,
            "lr":           scheduler.get_last_lr()[0],
            "best_val_L2":  best_val_l2
        })

        if epoch % 20 == 0:
            print(f"{geometry_name} | Epoch {epoch:3d} | "
                  f"Train L2: {train_l2:.4f} | Val L2: {val_l2:.4f} | "
                  f"Best: {best_val_l2:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    print(f"\n{geometry_name} done. Best Val L2: {best_val_l2:.4f}")
    wandb.finish()

In [ ]:
wandb.login()

for geometry_name, (x_path, y_path) in paths.items():
    train_geometry(geometry_name, x_path, y_path)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.



==== Training on NURBS ====


best_val_L2,█▆▆▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█████
lr,██████████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▁▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_rel_L2,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_rel_L2,█▃▃▂▂▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_L2,1.00088
epoch,155
lr,0.00014
train_loss,0.99858


NURBS | Epoch   0 | Train L2: 1.4522 | Val L2: 1.2022 | Best: 1.2022 | LR: 3.00e-04
NURBS | Epoch  20 | Train L2: 1.0105 | Val L2: 1.0131 | Best: 1.0100 | LR: 2.69e-04
NURBS | Epoch  40 | Train L2: 1.0066 | Val L2: 1.0079 | Best: 1.0048 | LR: 1.92e-04
NURBS | Epoch  60 | Train L2: 1.0032 | Val L2: 1.0059 | Best: 1.0005 | LR: 9.99e-05
NURBS | Epoch  80 | Train L2: 1.0000 | Val L2: 1.0034 | Best: 1.0005 | LR: 2.69e-05

NURBS done. Best Val L2: 0.9841


best_val_L2,█▅▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇██
lr,██████████▇▇▇▇▇▇▆▆▆▅▅▅▄▄▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train_loss,█▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▁
train_rel_L2,█▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▃▃▃▂▁
val_loss,█▆▄▄▄▄▄▄▄▃▃▃▃▃▄▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▃▁▁▃▃▇
val_rel_L2,█▆▄▄▃▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▃▄
best_val_L2,0.98415
epoch,99
lr,0.0
train_loss,0.43929


...

Harmonics | Epoch   0 | Train L2: 1.4753 | Val L2: 1.1780 | Best: 1.1780 | LR: 3.00e-04
Harmonics | Epoch  20 | Train L2: 1.0107 | Val L2: 1.0097 | Best: 1.0086 | LR: 2.69e-04
Harmonics | Epoch  40 | Train L2: 1.0066 | Val L2: 1.0050 | Best: 1.0047 | LR: 1.92e-04
Harmonics | Epoch  60 | Train L2: 1.0030 | Val L2: 1.0027 | Best: 1.0023 | LR: 9.99e-05
Harmonics | Epoch  80 | Train L2: 0.9972 | Val L2: 0.9994 | Best: 0.9994 | LR: 2.69e-05

Harmonics done. Best Val L2: 0.9947


best_val_L2,█▆▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇███
lr,███████▇▇▇▇▇▇▇▆▆▆▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
train_loss,██████████████████████████████▇▇▇▇▇▆▆▆▅▁
train_rel_L2,█▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▆▅▅▄▃▁
val_loss,█▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▄
val_rel_L2,▇▅▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▃█
best_val_L2,0.99473
epoch,99
lr,0.0
train_loss,0.33903



==== Training on Skelneton ====


Skelneton | Epoch   0 | Train L2: 1.4442 | Val L2: 1.2616 | Best: 1.2616 | LR: 3.00e-04
Skelneton | Epoch  20 | Train L2: 1.0070 | Val L2: 1.0080 | Best: 1.0080 | LR: 2.69e-04
Skelneton | Epoch  40 | Train L2: 1.0040 | Val L2: 1.0046 | Best: 1.0043 | LR: 1.92e-04
Skelneton | Epoch  60 | Train L2: 1.0020 | Val L2: 1.0026 | Best: 1.0024 | LR: 9.99e-05
Skelneton | Epoch  80 | Train L2: 1.0007 | Val L2: 1.0013 | Best: 1.0007 | LR: 2.69e-05

Skelneton done. Best Val L2: 0.9987


best_val_L2,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
lr,█████████▇▇▇▇▆▆▆▅▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁
train_loss,█▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁
train_rel_L2,█▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▃▃▂▁
val_loss,█▃▂▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▃
val_rel_L2,█▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▄▄
best_val_L2,0.99872
epoch,99
lr,0.0
train_loss,0.6096
